# Week 5 - Data Augmentation (RQ2)

## Overview
This notebook investigates whether back-translation data augmentation
improves AfriBERTa's performance on Hausa semantic relatedness prediction.

**Research Question 2:** Does augmenting the Hausa training data with
back-translated sentences improve model performance?

In [4]:
import torch
from transformers import AutoTokenizer, AutoModel
from torch import nn
from torch.utils.data import DataLoader, Dataset
from scipy.stats import spearmanr
import pandas as pd
import numpy as np
!pip install transformers==4.35.0

## Step 1 - Load Data
Loading the Hausa training set for augmentation and the Hausa test
set for final evaluation.dfHasTrain = pd.read_csv("hausa_train_clean.csv")
dfHasTest= pd.read_csv("hausa_test.csv")

print (dfHasTrain.columns.tolist())
print(f"Training pairs: {len(dfHasTrain)}")


In [5]:
HAS_TRAIN="https://raw.githubusercontent.com/Rosemary2301/SemRel-Hausa-Project/refs/heads/main/data/hausa_train_clean.csv"
HAS_TEST="https://raw.githubusercontent.com/Rosemary2301/SemRel-Hausa-Project/refs/heads/main/data/hausa_test.csv"




dfHasTest= pd.read_csv(HAS_TEST)
dfHasTrain = pd.read_csv(HAS_TRAIN)

print (dfHasTrain.columns.tolist())
print(f"Training pairs: {len(dfHasTrain)}")


['sentence1', 'sentence2', 'label']
Training pairs: 1736


## Step 2 - Back-Translation Pipeline
Using Facebook's NLLB model (No Language Left Behind) to back-translate
Hausa training sentences.

The process works as follows:
- Hausa sentence → translated to English → translated back to Hausa
- This creates new synthetic sentence pairs with slightly different wording
- The original similarity scores are kept as the labels remain the same

NLLB supports 200 languages including Hausa (language code: hau_Latn).

In [6]:
# Dataset class
class SemRelDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            row['sentence1'],
            row['sentence2'],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(row['label'], dtype=torch.float)
        }

# Model class
class SemRelModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.regressor = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        score = torch.sigmoid(self.regressor(cls_output))
        return score.squeeze()

!pip install transformers sentencepiece -q

from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

translator_ha_en = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="hau_Latn",
    tgt_lang="eng_Latn",
    device=device
)

translator_en_ha = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",
    tgt_lang="hau_Latn",
    device=device
)

def backTranslate(sentences, batch_size=16):
    backTrans = []
    for i, sent in enumerate(sentences):
        try:
            english = translator_ha_en(sent, max_length=256)[0]['translation_text']
            hausa = translator_en_ha(english, max_length=256)[0]['translation_text']
            backTrans.append(hausa if hausa else sent)
        except Exception as e:
            print(f"Error on sentence {i}: {e}")
            backTrans.append(sent)

        if (i + 1) % 50 == 0:
            print(f"Translated {i+1}/{len(sentences)}")

    return backTrans

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


## Step 3 - Build Augmented Dataset
Combining the original Hausa training data with the back-translated
synthetic data to create a larger training set.

- Original training pairs are kept as is
- Back-translated pairs are added on top
- The combined dataset gives AfriBERTa more examples to learn from

In [7]:
dfSample = dfHasTrain.reset_index(drop=True)
print(f"Translating {len(dfSample)} sentence pairs...")

print("Back-translating sentence1...")
btSent1 = backTranslate(dfSample['sentence1'].tolist())
print("Back-translating sentence2...")
btSent2 = backTranslate(dfSample['sentence2'].tolist())

dfAugmented = pd.DataFrame({
    "sentence1": btSent1,
    "sentence2": btSent2,
    "label":     dfSample['label'].values
})

dfCombined = pd.concat([dfHasTrain, dfAugmented], ignore_index=True)
dfCombined.to_csv("hausa_train_augmented.csv", index=False)

from google.colab import files
files.download("hausa_train_augmented.csv")

print(f"Original training pairs:  {len(dfHasTrain)}")
print(f"Augmented pairs created:  {len(dfAugmented)}")
print(f"Total combined pairs:     {len(dfCombined)}")

Translating 1736 sentence pairs...
Back-translating sentence1...


/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


Translated 50/1736
Translated 100/1736
Translated 150/1736
Translated 200/1736
Translated 250/1736
Translated 300/1736
Translated 350/1736
Translated 400/1736
Translated 450/1736
Translated 500/1736
Translated 550/1736
Translated 600/1736
Translated 650/1736
Translated 700/1736
Translated 750/1736
Translated 800/1736
Translated 850/1736
Translated 900/1736
Translated 950/1736
Translated 1000/1736
Translated 1050/1736
Translated 1100/1736
Translated 1150/1736
Translated 1200/1736
Translated 1250/1736
Translated 1300/1736
Translated 1350/1736
Translated 1400/1736
Translated 1450/1736
Translated 1500/1736
Translated 1550/1736
Translated 1600/1736
Translated 1650/1736
Translated 1700/1736
Back-translating sentence2...
Translated 50/1736
Translated 100/1736
Translated 150/1736
Translated 200/1736
Translated 250/1736
Translated 300/1736
Translated 350/1736
Translated 400/1736
Translated 450/1736
Translated 500/1736
Translated 550/1736
Translated 600/1736
Translated 650/1736
Translated 700/17

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Original training pairs:  1736
Augmented pairs created:  1736
Total combined pairs:     3472


## Step 4 - Train AfriBERTa on Augmented Data
Fine-tuning AfriBERTa (castorini/afriberta_large) on the combined
augmented dataset.

AfriBERTa was specifically pre-trained on African languages including
Hausa, making it better suited for this task than general multilingual
models like mBERT or LaBSE.


In [8]:
AFRIBERTA_MODEL = "castorini/afriberta_large"

print("Loading AfriBERTa tokenizer and model...")
Afritokenizer = AutoTokenizer.from_pretrained(AFRIBERTA_MODEL)
Afrimodel     = SemRelModel(AFRIBERTA_MODEL)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Afrimodel.to(device)

dfCombined = pd.read_csv("/content/hausa_train_augmented.csv")
augData    = SemRelDataset(dfCombined, Afritokenizer)
augLoader  = DataLoader(augData, batch_size=16, shuffle=True)

optimizer = torch.optim.AdamW(Afrimodel.parameters(), lr=2e-5)
lossFunc  = nn.MSELoss()

EPOCHS = 3
print("Starting AfriBERTa training on augmented data...")

for epoch in range(EPOCHS):
    Afrimodel.train()
    total_loss = 0
    for batch in augLoader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        preds = Afrimodel(input_ids, attention_mask)
        loss  = lossFunc(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(augLoader):.4f}")

# Save locally
torch.save(Afrimodel.state_dict(), "afriberta_augmented.pt")
print("Augmented AfriBERTa saved!")

Loading AfriBERTa tokenizer and model...


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:473: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Some weights of XLMRobertaModel were not initialized from the model checkpoint at castorini/afriberta_large and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting AfriBERTa training on augmented data...
Epoch 1 | Loss: 0.0618
Epoch 2 | Loss: 0.0381
Epoch 3 | Loss: 0.0238
Augmented AfriBERTa saved!


## Step 5 - Gain Analysis
Comparing the augmented AfriBERTa against the original AfriBERTa
trained in Week 3 to measure the impact of back-translation.

Both models are evaluated on the same Hausa test set using
Spearman Correlation so the comparison is fair.

In [9]:
def get_predictions(model, tokenizer, df, device, batch_size=16):
    dataset = SemRelDataset(df, tokenizer)
    loader  = DataLoader(dataset, batch_size=batch_size)
    model.eval()
    all_preds = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            preds          = model(input_ids, attention_mask)
            all_preds.extend(preds.cpu().numpy())

    return np.array(all_preds)

augmented_preds = get_predictions(Afrimodel, Afritokenizer, dfHasTest, device)

gold           = dfHasTest["label"].values
original_corr  = 0.4974
augmented_corr, _ = spearmanr(augmented_preds, gold)

print(f"""
--- Gain Analysis ---
AfriBERTa Original:   Spearman r = {original_corr:.4f}
AfriBERTa Augmented:  Spearman r = {augmented_corr:.4f}
Improvement:          {augmented_corr - original_corr:+.4f}
""")



--- Gain Analysis ---
AfriBERTa Original:   Spearman r = 0.4974
AfriBERTa Augmented:  Spearman r = 0.4760
Improvement:          -0.0214



## Results

| Model | Spearman Correlation |
|---|---|
| Dice Baseline | 0.4031 |
| AfriBERTa Original | 0.4974 |
| mBERT | 0.5947 |
| LaBSE Zero-Shot | 0.6443 |
| AfriBERTa Augmented | 0.4760|